# S6_4 — Confusion Matrix Deep Dive

**Leaf-JEPA IRP** | Stage 6 — Analysis and Interpretation


**Purpose:** Build DIFFERENTIAL confusion matrices showing what changed between models.
Cross-reference with Stage 2 predicted confusion pairs.

**Outputs:**
- `confusion_diff_B5_minus_B3.png` — domain pretraining impact
- `confusion_diff_PEFT_minus_LP.png` — PEFT adaptation impact
- `confusion_improvement_table.csv` — top improved and worsened pairs
- `confusion_pair_persistence.csv` — which confusions persist across all methods

**Research Questions Served:** RQ1 (which confusions does domain PT resolve?), RQ4


## Initialization

In [1]:

import sys, json
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("..").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from stage6_analysis_and_interpretation.config_stage6 import *
from stage6_analysis_and_interpretation.analysis_utils import (
    set_seed, load_json, save_json,
    compute_normalised_cm, confusion_difference,
    plot_confusion_diff, top_confusion_changes,
    rank_classes_by_difficulty, analyse_difficulty_factors,
    print_section,
)
import matplotlib.pyplot as plt
import seaborn as sns

set_seed(RANDOM_SEED)
STAGE6_FIGURES.mkdir(parents=True, exist_ok=True)
STAGE6_TABLES.mkdir(parents=True, exist_ok=True)


## Load predictions / confusion matrices

In [ ]:
print_section("Loading Predictions")

# Load class names
split_data = load_json(SPLITS_DIR / "test.json")
class_names = split_data.get("class_names", [f"class_{i}" for i in range(NUM_CLASSES)])

# Try to load saved predictions or confusion matrices
# Adjust paths based on your actual Stage 3/5 output format
cm_data = {}

for bid in BASELINE_IDS:
    # Try loading saved confusion matrix
    cm_path = BASELINES_DIR / f"{bid}_confusion_matrix.npy"
    pred_path = BASELINES_DIR / f"{bid}_predictions.json"
    agg_path = BASELINES_DIR / f"{bid}_aggregate.json"

    if cm_path.exists():
        cm_data[bid] = np.load(cm_path)
        print(f"  {bid}: loaded CM from .npy")
    elif pred_path.exists():
        preds = load_json(pred_path)
        y_true = np.array(preds["y_true"])
        y_pred = np.array(preds["y_pred"])
        cm_data[bid] = compute_normalised_cm(y_true, y_pred, NUM_CLASSES)
        print(f"  {bid}: computed CM from predictions")
    elif agg_path.exists():
        agg = load_json(agg_path)
        if "confusion_matrix" in agg:
            cm_data[bid] = np.array(agg["confusion_matrix"])
            # Normalise if raw counts
            row_sums = cm_data[bid].sum(axis=1, keepdims=True)
            row_sums[row_sums == 0] = 1
            cm_data[bid] = cm_data[bid] / row_sums
            print(f"  {bid}: extracted CM from aggregate JSON")
    else:
        print(f"  \u26a0\ufe0f {bid}: no CM data found")

# Also load PEFT confusion matrices
for method in PEFT_METHODS:
    cm_path = PEFT_RESULTS_DIR / f"{method}_confusion_matrix.npy"
    pred_path = PEFT_RESULTS_DIR / f"{method}_predictions.json"
    if cm_path.exists():
        cm_data[method] = np.load(cm_path)
        print(f"  {method}: loaded CM")
    elif pred_path.exists():
        preds = load_json(pred_path)
        cm_data[method] = compute_normalised_cm(
            np.array(preds["y_true"]), np.array(preds["y_pred"]), NUM_CLASSES)
        print(f"  {method}: computed CM")

print(f"\n  Available CMs: {list(cm_data.keys())}")


## Differential Confusion matrices

In [ ]:
print_section("Differential Confusion Matrices")

# B5 - B3 = domain pretraining contribution
if "B3" in cm_data and "B5" in cm_data:
    diff_b5_b3 = confusion_difference(cm_data["B3"], cm_data["B5"])
    plot_confusion_diff(diff_b5_b3, class_names,
                        STAGE6_FIGURES / "confusion_diff_B5_minus_B3.png",
                        title="Confusion Change: Leaf-JEPA LP minus Generic I-JEPA LP")

    improved, worsened = top_confusion_changes(diff_b5_b3, class_names, n=10)
    print("  Top 10 IMPROVED confusion pairs (domain pretraining):")
    for item in improved:
        print(f"    {item['true_class']} -> {item['pred_class']}: {item['change']:+.4f}")
    print("\n  Top 10 WORSENED confusion pairs:")
    for item in worsened:
        print(f"    {item['true_class']} -> {item['pred_class']}: {item['change']:+.4f}")

    # Save
    pd.DataFrame(improved + worsened).to_csv(
        STAGE6_TABLES / "confusion_improvement_B5_vs_B3.csv", index=False)
else:
    print("  \u26a0\ufe0f Need both B3 and B5 CMs for domain pretraining differential")

# Best PEFT - B5 LP = PEFT adaptation contribution
best_peft = None
for method in PEFT_METHODS:
    if method in cm_data:
        best_peft = method
        break

if best_peft and "B5" in cm_data:
    diff_peft_lp = confusion_difference(cm_data["B5"], cm_data[best_peft])
    plot_confusion_diff(diff_peft_lp, class_names,
                        STAGE6_FIGURES / "confusion_diff_PEFT_minus_LP.png",
                        title=f"Confusion Change: {best_peft} minus Leaf-JEPA LP")


## Confusion pair persistence

In [ ]:
print_section("Confusion Pair Persistence Across Methods")

# For each off-diagonal cell, check if it's consistently high across all methods
persistence_data = []
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if i == j:
            continue
        values = {}
        for method, cm in cm_data.items():
            if cm.shape[0] == NUM_CLASSES:
                values[method] = float(cm[i, j])
        if values:
            avg_conf = np.mean(list(values.values()))
            if avg_conf > 0.02:  # Only track non-trivial confusions
                persistence_data.append({
                    "true_class": class_names[i],
                    "pred_class": class_names[j],
                    "avg_confusion": avg_conf,
                    "n_methods": len(values),
                    "persistent": all(v > 0.01 for v in values.values()),
                    **values,
                })

persistence_df = pd.DataFrame(persistence_data).sort_values("avg_confusion", ascending=False)
persistence_df.to_csv(STAGE6_TABLES / "confusion_pair_persistence.csv", index=False)

print(f"  Total non-trivial confusion pairs: {len(persistence_df)}")
print(f"  Persistent across ALL methods: {persistence_df['persistent'].sum()}")
print("\n  Top 10 persistent confusions:")
print(persistence_df.head(10)[["true_class", "pred_class", "avg_confusion", "persistent"]].to_string(index=False))

# Cross-reference with Stage 2 predictions
s2_pairs_path = ANALYSIS_DIR / "top_confused_pairs.json" if ANALYSIS_DIR else None
if s2_pairs_path and s2_pairs_path.exists():
    predicted = load_json(s2_pairs_path)
    predicted_pairs = set()
    for p in predicted[:10]:
        predicted_pairs.add((p.get("class_a", ""), p.get("class_b", "")))
        predicted_pairs.add((p.get("class_b", ""), p.get("class_a", "")))

    actual_top = set()
    for _, row in persistence_df.head(10).iterrows():
        actual_top.add((row["true_class"], row["pred_class"]))

    overlap = predicted_pairs & actual_top
    print(f"\n  Stage 2 predicted {len(predicted[:10])} confused pairs")
    print(f"  Actual top 10: {len(actual_top)} pairs")
    print(f"  Overlap: {len(overlap)} pairs")
    if overlap:
        print(f"  Confirmed predictions: {overlap}")


## Per class F1 ranking (hard classes)

In [ ]:
print_section("Hard Class Ranking")

# Collect per-class F1 from all methods
per_class_f1 = {}
for method_name in list(cm_data.keys()):
    agg_path = None
    if method_name.startswith("B"):
        agg_path = BASELINES_DIR / f"{method_name}_aggregate.json"
    else:
        agg_path = PEFT_RESULTS_DIR / f"{method_name}_aggregate.json"

    if agg_path and agg_path.exists():
        agg = load_json(agg_path)
        if "per_class_f1" in agg:
            per_class_f1[method_name] = agg["per_class_f1"]
    elif method_name in cm_data:
        # Compute from confusion matrix diagonal (recall)
        cm = cm_data[method_name]
        per_class_f1[method_name] = [float(cm[i, i]) for i in range(min(NUM_CLASSES, cm.shape[0]))]

if per_class_f1:
    hard_df = rank_classes_by_difficulty(per_class_f1, class_names[:len(list(per_class_f1.values())[0])])
    hard_df.to_csv(STAGE6_TABLES / "hard_classes_analysis.csv")

    print("  Hardest classes (lowest avg F1 across all methods):")
    print(hard_df.head(10)[["avg_f1", "std_f1"]].to_string())

    # Per-class F1 heatmap
    fig, ax = plt.subplots(figsize=(max(8, len(per_class_f1) * 2), 16))
    plot_df = hard_df.drop(columns=["avg_f1", "std_f1"])
    sns.heatmap(plot_df, annot=True, fmt=".2f", cmap="RdYlGn", vmin=0, vmax=1,
                ax=ax, linewidths=0.3, annot_kws={"fontsize": 6})
    ax.set_title("Per-Class F1 Across All Methods (sorted by difficulty)", fontsize=13)
    plt.yticks(fontsize=7)
    plt.xticks(fontsize=9, rotation=45, ha="right")
    plt.tight_layout()
    fig.savefig(STAGE6_FIGURES / "per_class_f1_heatmap_all_methods.png")
    plt.close(fig)
    print("  Heatmap saved.")
else:
    print("  \u26a0\ufe0f No per-class F1 data available")

print("\n\u2705 S6_4 complete.")


## Critical Analysis

### Differential Confusion Matrices
- Negative values (blue) = confusions that DECREASED = improvements from domain pretraining
- Positive values (red) = confusions that INCREASED = regressions
- A mostly-blue difference matrix is strong evidence for domain pretraining value

### Stage 2 Cross-Reference
If the top persistent confusions match the Stage 2 predicted pairs (cosine similarity > 0.9),
write in the dissertation: "As predicted by the pre-training inter-class similarity analysis,
the highest-confusion pairs were X and Y, confirming that model errors are driven by genuine
visual ambiguity rather than insufficient training."

### Pathogen Category Confusions
Analyse whether confusions are more common WITHIN pathogen categories (two fungal diseases
confused) or ACROSS categories (fungal confused with bacterial). Within-category confusions
are biologically expected and less severe; cross-category confusions suggest the model hasn't
learned disease-type features.
